In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig, DataCollatorForLanguageModeling
import pandas as pd
import matplotlib.pyplot as plt
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig

In [1]:
!pip install transformers==4.53.3 trl==0.12.0 #poi riavvia

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install -U bitsandbytes #poi riavvia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00


Decidiamo che tipo ddi model e dataset vogliamo usare. I modelli che io voglio usare per effettuare un confronto delle prestazioni sono :
- Phi-4-mini-instruct 4B
- Llama 3 8B
- Gemma-2-2B
Questo confronto permette anche di coprire 3 diverse taglie

In [2]:
training_config = SFTConfig(
    bf16=False,                # Cambia in True SOLO se hai A100/L4
    fp16=True,                 # Lascia True per la T4 gratuita
    do_eval=True,              # Importante per la tesi mostrare i grafici di validazione
    eval_strategy="steps",
    eval_steps = 50,           # Valuta a fine ogni epoca
    learning_rate=5e-5,        # Un valore equilibrato
    log_level="info",
    logging_steps=10,
    lr_scheduler_type="cosine",
    num_train_epochs=3,        # 3 epoche è lo standard per il fine-tuning
    output_dir="/content/drive/MyDrive/phi4_checkpoints", # Salva su Drive!
    overwrite_output_dir=True,
    per_device_train_batch_size=1,
    eval_accumulation_steps=1,
    gradient_accumulation_steps=8, # 2 (batch) * 8 (accumulo) = 16
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    warmup_ratio=0.1,
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_steps=50,
    save_total_limit=2,
    seed=0,
    optim="paged_adamw_32bit",
    report_to="none",
)

In [3]:
model_name = "microsoft/Phi-4-mini-instruct"
#model_name= "meta-llama/Meta-Llama-3-8B"
#model_name= "google/gemma-2-2b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_quant_type= 'nf4',
    bnb_4bit_compute_dtype= torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config, #ci permette di quantizzare il modello
    device_map = 'auto', #off load stuff to cpu if you don't have enough memory
    trust_remote_code = True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fondamentale per il training

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
#per phi
from peft import prepare_model_for_kbit_training

# 1. Prepara il modello per il training quantizzato
model = prepare_model_for_kbit_training(model)

# 2. (Opzionale ma consigliato) Forza i parametri a richiedere i gradienti
model.gradient_checkpointing_enable()

In [5]:
peft_config = {
    "r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM",
    "target_modules": "all-linear",
    "modules_to_save": None,
}

peft_conf = LoraConfig(**peft_config)

In [7]:
ds = load_dataset("FMiMiY/SS-GEN")

Testiamo le prestazioni dei modelli su tre modalità:
1) Zero-shot: interroghiamo i modelli afficnhé generino il contenuto della storia a partire dal titolo;
2) Few-shot: diamo solo un certo numero di esempi;
3) Fine-tuning: utilizziamo il dataset proposto e poi confrontiamo i risultati dei vari modelli.

In [8]:
#fine-tuning
def tokenizer_ft(batch):

    #template riportato dalla figura 5 del paper
    prompt = (
        "Develop a concise, clear, straightforward, positive and supportive "
        "Social Story titled \"{title}\" for children and teens with autism, "
        "200-300 words, that promotes their social understanding and boosts "
        "their participation in daily activities, fostering independence and confidence."
    )

    texts = [
        f"{prompt.format(title=t)}\nTitle: {t}\n\nSocial Story: {s}{tokenizer.eos_token}"
        for t,s in zip(batch['title'], batch['story_content'])
    ]
    tokens = tokenizer(
        texts,
        padding = "max_length",
        truncation = True,
        max_length = 512,
        return_tensors = None
    )

    tokens['labels'] = tokens["input_ids"].copy()

    return tokens

In [9]:
tokenized_data = ds.map(tokenizer_ft,batched=True, remove_columns=ds["train"].column_names)

Map:   0%|          | 0/509 [00:00<?, ? examples/s]

In [10]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    dataset_text_field="story_content",  # La colonna dove hai le tue Social Stories
    max_seq_length=512,         # Taglia i testi troppo lunghi
    args=training_config,                  # I tuoi TrainingArguments di prima
    peft_config=peft_conf,    # Passi la configurazione LoRA direttamente qui
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not giv

In [ ]:
#trainer.train(resume_from_checkpoint=True) #se l'ho fatto partire
trainer.train()
#salva la cronologia dell'addestramento anche per ottenere il grafico
# Estrai i log
history = trainer.state.log_history
df = pd.DataFrame(history)

train_df = df[df['loss'].notna()]
eval_df = df[df['eval_loss'].notna()]

# Salva i log in un CSV su Drive per la tesi
df.to_csv("/content/drive/MyDrive/training_phi4_logs.csv", index=False)
print("Log salvati su Drive: training_phi4_logs.csv")

# Crea un grafico veloce della Loss
plt.figure(figsize=(12, 6))
# Plot Training Loss
plt.plot(train_df['step'], train_df['loss'], label='Training Loss', color='blue', alpha=0.7)

# Plot Validation Loss
if not eval_df.empty:
    plt.plot(eval_df['step'], eval_df['eval_loss'], label='Validation Loss', color='red', marker='o', linestyle='--')

# Formattazione estetica
plt.title('Andamento della Loss: Training vs Validation', fontsize=14)
plt.xlabel('Step di Addestramento', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

# 6. Salvataggio e visualizzazione
plt.savefig("/content/drive/MyDrive/loss_chart_tesi.png", dpi=300) # Alta risoluzione per la tesi
plt.show()

print("Grafico salvato su Drive: loss_chart_tesi.png")

***** Running training *****
  Num examples = 4,068
  Num Epochs = 3
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 8
  Total optimization steps = 1,527
  Number of trainable parameters = 11,534,336


Step,Training Loss,Validation Loss
50,1.361800,1.372791
100,1.174500,1.164836
150,1.129900,1.084685
200,1.066800,1.025670
250,0.977900,0.990291
300,0.957100,0.969821
350,1.003000,0.957337
400,0.977500,0.949431
450,0.928300,0.940955
500,0.976500,0.935144



***** Running Evaluation *****
  Num examples = 509
  Batch size = 8
Saving model checkpoint to /content/drive/MyDrive/phi4_checkpoints/checkpoint-50
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--microsoft--Phi-4-mini-instruct/snapshots/cfbefacb99257ffa30c83adab238a50856ac3083/config.json
Model config Phi3Config {
  "architectures": [
    "Phi3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "configuration_phi3.Phi3Config",
    "AutoModelForCausalLM": "modeling_phi3.Phi3ForCausalLM",
    "AutoTokenizer": "Xenova/gpt-4o"
  },
  "bos_token_id": 199999,
  "embd_pdrop": 0.0,
  "eos_token_id": 199999,
  "full_attn_mod": 1,
  "hidden_act": "silu",
  "hidden_size": 3072,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "interpolate_factor": 1,
  "lm_head_bias": false,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "phi3",
  "num_attention_heads": 24,
  "

Step,Training Loss,Validation Loss
50,1.361800,1.372791
100,1.174500,1.164836
150,1.129900,1.084685
200,1.066800,1.025670
250,0.977900,0.990291
300,0.957100,0.969821
350,1.003000,0.957337
400,0.977500,0.949431
450,0.928300,0.940955
500,0.976500,0.935144



***** Running Evaluation *****
  Num examples = 509
  Batch size = 8
Saving model checkpoint to /content/drive/MyDrive/phi4_checkpoints/checkpoint-550
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--microsoft--Phi-4-mini-instruct/snapshots/cfbefacb99257ffa30c83adab238a50856ac3083/config.json
Model config Phi3Config {
  "architectures": [
    "Phi3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "configuration_phi3.Phi3Config",
    "AutoModelForCausalLM": "modeling_phi3.Phi3ForCausalLM",
    "AutoTokenizer": "Xenova/gpt-4o"
  },
  "bos_token_id": 199999,
  "embd_pdrop": 0.0,
  "eos_token_id": 199999,
  "full_attn_mod": 1,
  "hidden_act": "silu",
  "hidden_size": 3072,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "interpolate_factor": 1,
  "lm_head_bias": false,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "phi3",
  "num_attention_heads": 24,
  

In [ ]:
# Definisci il percorso di salvataggio su Drive
output_dir = "/content/drive/MyDrive/phi4_ss_gen_final"

# 1. Salva il modello (solo i pesi LoRA / PEFT adapters)
trainer.model.save_pretrained(output_dir)

# 2. Salva anche il tokenizer (fondamentale per ricaricarlo dopo)
tokenizer.save_pretrained(output_dir)

print(f"Modello e Tokenizer salvati correttamente in: {output_dir}")

In [ ]:
#Per usare il modello fine-tuned per generare storie
# Percorsi
base_model_name = "microsoft/Phi-4-mini-instruct"
adapter_path = "/content/drive/MyDrive/phi4_ss_gen_final"

# 1. Carica il modello base (lo stesso usato per il training)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# 2. Carica il Tokenizer salvato
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# 3. Carica gli adapter (i pesi allenati da te) sopra il modello base
model = PeftModel.from_pretrained(base_model, adapter_path)

# Ora il 'model' è quello intelligente che sa scrivere Social Stories!
model.eval()